# 06 – Silver MERGE

Applies the CDC batches in landing to the silver tables with MERGE. The steps are shown on the card table; the runner then applies all 4 tables.

## Import

Adds the `src` folder to the Python path and loads the `silver_transforms` and `silver_merge` modules.

**Note:** Re-run this cell after a `.py` file changes (`importlib.reload`).

In [0]:
import os
import sys
import importlib

from pyspark.sql import functions as F
from pyspark.sql import Window

# Notebook is under notebooks/; src is at the repo root
repo_root = os.path.dirname(os.getcwd())
src_path = os.path.join(repo_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

import silver_transforms
import silver_merge

# silver_merge uses silver_transforms: reload that one first
importlib.reload(silver_transforms)
importlib.reload(silver_merge)

print("src_path:", src_path)
print("CDC_TABLES:", silver_transforms.CDC_TABLES)

## Read a landing batch

Reads card batch 2 with `silver_merge.read_batch` (explicit schema, `change_ts` as timestamp).

**Expected output:** `op_type` counts I 5, U 9, D 2 (16 in total) and `null_change_ts` 0.

In [0]:
df_card_batch = silver_merge.read_batch(spark, "card", 2)

df_card_batch.groupBy("op_type").count().orderBy("op_type").show()

null_change_ts = df_card_batch.filter(F.col("change_ts").isNull()).count()
print("rows:", df_card_batch.count(), "| null_change_ts:", null_change_ts)

## Dedup

Keeps one row per key within the batch. If more than one source row matches the same target row, MERGE fails.

| Rule | Result |
|---|---|
| Latest `change_ts` per key | `ROW_NUMBER` → only `rn = 1` is kept |
| Key has an I in the batch (`has_insert`) and the last op is not D | `op_type` = I (the card is not in the target yet; it is inserted with its latest values) |

**Expected output:** 16 rows → 15 rows / 15 keys. `card_id` 1253: one row, `op_type` I, `type` gold.

In [0]:
# Latest record per key. SQL: ROW_NUMBER() OVER (PARTITION BY card_id ORDER BY change_ts DESC)
window_latest = Window.partitionBy("card_id").orderBy(F.desc("change_ts"))

# Does the key have an I in this batch. SQL: MAX(CASE WHEN op_type = 'I' THEN 1 ELSE 0 END) OVER (PARTITION BY card_id)
window_key = Window.partitionBy("card_id")
is_insert = F.when(F.col("op_type") == "I", F.lit(1)).otherwise(F.lit(0))

df_card_dedup = (
    df_card_batch
    .withColumn("rn", F.row_number().over(window_latest))
    .withColumn("has_insert", F.max(is_insert).over(window_key))
    .filter(F.col("rn") == 1)
    # I → U: the card is not in the target yet, insert it with its latest values
    .withColumn("op_type", F.when((F.col("has_insert") == 1) & (F.col("op_type") != "D"), "I").otherwise(F.col("op_type")))
    .drop("rn", "has_insert")
)

df_card_dedup.filter(F.col("card_id") == "1253").show()
print("rows:", df_card_dedup.count(), "| distinct keys:", df_card_dedup.select("card_id").distinct().count())
df_card_dedup.groupBy("op_type").count().show()

## Prepare source and check target

Casts the dedup result to silver types, joins the CDC columns back on the key (`df_card_source`), and shows whether each source key exists in the target (`in_target`).

**Note:** Silver is already in its post-batch-3 state, so the `in_target` counts can differ from the first run. This section only reads; it writes nothing.

In [0]:
# to_silver drops the CDC columns; keep them separately
df_card_data = silver_transforms.to_silver(df_card_dedup, "card")
df_card_cdc = df_card_dedup.select(
    F.col("card_id").cast("int").alias("card_id"),
    "op_type",
    "change_ts",
    "batch_id",
)

# Safe join: one row per key after dedup
df_card_source = df_card_data.join(df_card_cdc, on="card_id", how="inner")

# TARGET: existing keys in silver
df_card_target = spark.table("workspace.silver.card")
df_target_keys = df_card_target.select("card_id").withColumn("in_target", F.lit(True))

# Left join: in_target is NULL for keys not in the target → false
df_check = df_card_source.join(df_target_keys, on="card_id", how="left")
df_check = df_check.withColumn("in_target", F.coalesce(F.col("in_target"), F.lit(False)))

df_check.groupBy("op_type", "in_target").count().orderBy("op_type", "in_target").show()
print("target rows:", df_card_target.count())

## Card MERGE SQL

Shows the card MERGE used by the runner. The SQL is in the `MERGE_SQL` dictionary in `src/silver_merge.py`.

| Clause | Condition | Action |
|---|---|---|
| WHEN MATCHED | `op_type = 'D'` | DELETE |
| WHEN MATCHED | `op_type = 'U'` and `s.change_ts > t._change_ts` | UPDATE |
| WHEN NOT MATCHED | `op_type = 'I'` | INSERT |

Rows that match no clause are skipped; U / D records for keys not in the target are written to quarantine by the runner.

**Expected output:** SQL text. This cell does not run the MERGE.

In [0]:
print(silver_merge.MERGE_SQL["card"])

## Batch log and quarantine tables

`workspace.silver.batch_log`: one row per (table, batch); status STARTED / SUCCESS / FAILED. A batch already marked SUCCESS does not go into MERGE again.

`workspace.silver.quarantine`: U / D records for keys not in the target; their count is `rows_skipped` in the log.

**Expected output:** `batch_log` and `quarantine` in the `SHOW TABLES` list. Safe to re-run (`IF NOT EXISTS`).

In [0]:
%sql
-- Batch-level protection: key = table_name + batch_id
CREATE TABLE IF NOT EXISTS workspace.silver.batch_log (
  table_name    STRING,
  batch_id      INT,
  status        STRING,
  rows_read     BIGINT,
  rows_inserted BIGINT,
  rows_updated  BIGINT,
  rows_deleted  BIGINT,
  rows_skipped  BIGINT,
  started_at    TIMESTAMP,
  finished_at   TIMESTAMP
);

-- U / D records for keys not in the target
CREATE TABLE IF NOT EXISTS workspace.silver.quarantine (
  table_name STRING,
  batch_id   INT,
  key_value  INT,
  op_type    STRING,
  reason     STRING,
  change_ts  TIMESTAMP
);

SHOW TABLES IN workspace.silver;

## Runner: apply batches with log protection

Applies the batches in `RUN_BATCH_IDS` in order (1 → 2 → 3); for each batch, all 4 tables with `silver_merge.run_batch`.

Flow: log check (SUCCESS → SKIPPED) → STARTED → quarantine (`left_anti`) → MERGE → SUCCESS (error → FAILED).

**Expected output:** one SUCCESS or SKIPPED line per (table, batch), followed by `batch_log`. If all 12 batches are already SUCCESS, all 12 lines are SKIPPED.

In [0]:
RUN_BATCH_IDS = [1, 2, 3]

# Outer loop: batch order; inner loop: tables
for batch_id in RUN_BATCH_IDS:
    for table_name in silver_transforms.CDC_TABLES:
        silver_merge.run_batch(spark, table_name, batch_id)

display(spark.sql(f"""
    SELECT table_name, batch_id, status, rows_read, rows_updated, rows_deleted,
           rows_inserted, rows_skipped, started_at, finished_at
    FROM {silver_merge.LOG_TABLE}
    ORDER BY batch_id, table_name
"""))

## Summary

Final state of the silver tables. `changed_rows` = rows with `_change_ts` > 1998-12-31, i.e. written by MERGE at least once.

**Expected output (after batch 3):** client 5,383 / 46, account 4,500 / 36, card 890 / 29, loan 682 / 24; `max_change_ts` in March 1999.

In [0]:
%sql
SELECT 'client' AS table_name, COUNT(*) AS target_rows,
       COUNT_IF(_change_ts > TIMESTAMP '1998-12-31 00:00:00') AS changed_rows,
       MAX(_change_ts) AS max_change_ts
FROM workspace.silver.client
UNION ALL
SELECT 'account', COUNT(*), COUNT_IF(_change_ts > TIMESTAMP '1998-12-31 00:00:00'), MAX(_change_ts)
FROM workspace.silver.account
UNION ALL
SELECT 'card', COUNT(*), COUNT_IF(_change_ts > TIMESTAMP '1998-12-31 00:00:00'), MAX(_change_ts)
FROM workspace.silver.card
UNION ALL
SELECT 'loan', COUNT(*), COUNT_IF(_change_ts > TIMESTAMP '1998-12-31 00:00:00'), MAX(_change_ts)
FROM workspace.silver.loan;